In [1]:
import torch
import traceback

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PASS = "[PASS]"
FAIL = "[FAIL]"


def section(title):
    print(f"\n{'='*50}")
    print(f"  {title}")
    print(f"{'='*50}")

Device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM: 6.1 GB


In [2]:
section("1. ContinuumRemoval")
try:
    from preprocessing.continuum_removal import ContinuumRemoval
    layer = ContinuumRemoval()
    x = torch.randn(4, 200, 16, 16).to(DEVICE)
    layer = layer.to(DEVICE)
    out = layer(x)
    assert out.shape == x.shape, f"Shape mismatch: {out.shape}"
    print(f"{PASS} Input: {tuple(x.shape)} -> Output: {tuple(out.shape)}")
except Exception:
    print(f"{FAIL}"); traceback.print_exc()


  1. ContinuumRemoval
[PASS] Input: (4, 200, 16, 16) -> Output: (4, 200, 16, 16)


In [3]:
section("2. Spectral3DStack")
try:
    from models.spectral_3d_block import Spectral3DStack
    stack = Spectral3DStack(num_filters=8, num_blocks=3).to(DEVICE)
    x = torch.randn(4, 200, 16, 16).to(DEVICE)
    out = stack(x)
    assert out.shape == (4, 8, 200, 16, 16), f"Shape mismatch: {out.shape}"
    print(f"{PASS} Input: {tuple(x.shape)} -> Output: {tuple(out.shape)}")
    print(f"      out_channels: {stack.out_channels}")
except Exception:
    print(f"{FAIL}"); traceback.print_exc()


  2. Spectral3DStack
[PASS] Input: (4, 200, 16, 16) -> Output: (4, 8, 200, 16, 16)
      out_channels: 8


In [4]:
section("3. SpectralTransformer (200 bands, B=4, 16x16 patch)")
try:
    from models.real_spectral_transformer import SpectralTransformer
    transformer = SpectralTransformer(embed_dim=8, num_heads=4, num_layers=2).to(DEVICE)
    x = torch.randn(4, 8, 200, 16, 16).to(DEVICE)

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    out = transformer(x)

    if torch.cuda.is_available():
        peak_mb = torch.cuda.max_memory_allocated() / 1e6
        print(f"      Peak VRAM: {peak_mb:.1f} MB")

    assert out.shape == (4, 8, 16, 16), f"Shape mismatch: {out.shape}"
    print(f"{PASS} Input: {tuple(x.shape)} -> Output: {tuple(out.shape)}")
except Exception:
    print(f"{FAIL}"); traceback.print_exc()


  3. SpectralTransformer (200 bands, B=4, 16x16 patch)
      Peak VRAM: 3849.5 MB
[PASS] Input: (4, 8, 200, 16, 16) -> Output: (4, 8, 16, 16)


In [5]:
section("4. SpectralTransformer — band agnosticism (103 bands)")
try:
    from models.real_spectral_transformer import SpectralTransformer
    transformer = SpectralTransformer(embed_dim=8, num_heads=4, num_layers=2).to(DEVICE)
    x = torch.randn(1, 8, 103, 16, 16).to(DEVICE)
    out = transformer(x)
    assert out.shape == (1, 8, 16, 16), f"Shape mismatch: {out.shape}"
    print(f"{PASS} Input: {tuple(x.shape)} -> Output: {tuple(out.shape)}")
    print(f"      Positional embeddings interpolated 500 -> 103 bands. No crash.")
except Exception:
    print(f"{FAIL}"); traceback.print_exc()


  4. SpectralTransformer — band agnosticism (103 bands)
[PASS] Input: (1, 8, 103, 16, 16) -> Output: (1, 8, 16, 16)
      Positional embeddings interpolated 500 -> 103 bands. No crash.


In [6]:
section("5. Encoder2D")
try:
    from models.encoder_2d import Encoder2D
    enc = Encoder2D(in_channels=8, base_filters=32).to(DEVICE)
    x = torch.randn(4, 8, 16, 16).to(DEVICE)
    b, skips = enc(x)
    print(f"{PASS} Input: {tuple(x.shape)}")
    print(f"      Bottleneck: {tuple(b.shape)}")
    for i, s in enumerate(skips):
        print(f"      Skip {i+1}: {tuple(s.shape)}")
except Exception:
    print(f"{FAIL}"); traceback.print_exc()


  5. Encoder2D
[PASS] Input: (4, 8, 16, 16)
      Bottleneck: (4, 512, 1, 1)
      Skip 1: (4, 32, 16, 16)
      Skip 2: (4, 64, 8, 8)
      Skip 3: (4, 128, 4, 4)
      Skip 4: (4, 256, 2, 2)


In [7]:
section("6. Decoder2D")
try:
    from models.decoder_2d import Decoder2D
    f = 32
    dec = Decoder2D(num_classes=17, base_filters=f).to(DEVICE)
    b = torch.randn(4, f * 16, 1, 1).to(DEVICE)
    skips = [
        torch.randn(4, f,     16, 16).to(DEVICE),
        torch.randn(4, f * 2,  8,  8).to(DEVICE),
        torch.randn(4, f * 4,  4,  4).to(DEVICE),
        torch.randn(4, f * 8,  2,  2).to(DEVICE),
    ]
    out = dec(b, skips)
    assert out.shape == (4, 17, 16, 16), f"Shape mismatch: {out.shape}"
    print(f"{PASS} Output: {tuple(out.shape)}")
except Exception:
    print(f"{FAIL}"); traceback.print_exc()


  6. Decoder2D
[PASS] Output: (4, 17, 16, 16)


In [8]:
section("7. Full Pipeline — 200 bands, B=4, 16x16")
try:
    from models.hyperspectral_net import HyperspectralNet
    model = HyperspectralNet(num_bands=200, num_classes=17, base_filters=32).to(DEVICE)
    x = torch.randn(4, 200, 16, 16).to(DEVICE)

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    with torch.no_grad():
        out = model(x)

    if torch.cuda.is_available():
        peak_mb = torch.cuda.max_memory_allocated() / 1e6
        print(f"      Peak VRAM: {peak_mb:.1f} MB")

    assert out.shape == (4, 17, 16, 16), f"Shape mismatch: {out.shape}"
    total_params = sum(p.numel() for p in model.parameters())
    print(f"{PASS} Input: {tuple(x.shape)} -> Output: {tuple(out.shape)}")
    print(f"      Parameters: {total_params:,}")
except Exception:
    print(f"{FAIL}"); traceback.print_exc()


  7. Full Pipeline — 200 bands, B=4, 16x16
      Peak VRAM: 2259.6 MB
[PASS] Input: (4, 200, 16, 16) -> Output: (4, 17, 16, 16)
      Parameters: 7,782,337


In [9]:
section("8. Full Pipeline — zero-shot 103 bands (Pavia)")
try:
    from models.hyperspectral_net import HyperspectralNet
    model = HyperspectralNet(num_bands=200, num_classes=17, base_filters=32).to(DEVICE)
    x = torch.randn(1, 103, 16, 16).to(DEVICE)

    with torch.no_grad():
        out = model(x)

    assert out.shape == (1, 17, 16, 16), f"Shape mismatch: {out.shape}"
    print(f"{PASS} Input: {tuple(x.shape)} -> Output: {tuple(out.shape)}")
    print(f"      Model trained on 200 bands handled 103 bands without retraining.")
except Exception:
    print(f"{FAIL}"); traceback.print_exc()


  8. Full Pipeline — zero-shot 103 bands (Pavia)
[PASS] Input: (1, 103, 16, 16) -> Output: (1, 17, 16, 16)
      Model trained on 200 bands handled 103 bands without retraining.


In [10]:
section("9. Backward Pass")
try:
    from models.hyperspectral_net import HyperspectralNet
    model = HyperspectralNet(num_bands=200, num_classes=17, base_filters=32).to(DEVICE)
    x      = torch.randn(4, 200, 16, 16).to(DEVICE)
    labels = torch.randint(0, 17, (4, 16, 16)).to(DEVICE)

    out  = model(x)
    loss = torch.nn.CrossEntropyLoss(ignore_index=0)(out, labels)
    loss.backward()

    dead = [n for n, p in model.named_parameters() if p.grad is None]
    if dead:
        print(f"{FAIL} Params with no gradient: {dead}")
    else:
        print(f"{PASS} Loss: {loss.item():.4f}. All parameters received gradients.")
except Exception:
    print(f"{FAIL}"); traceback.print_exc()


print(f"\n{'='*50}")
print("  Done. Fix any FAILs before running train.py.")
print(f"{'='*50}\n")


  9. Backward Pass
[PASS] Loss: 2.9018. All parameters received gradients.

  Done. Fix any FAILs before running train.py.



In [1]:
import torch
from models.hyperspectral_net import HyperspectralNet

DEVICE = torch.device('cuda')
model = HyperspectralNet(num_bands=200, num_classes=17, base_filters=32).to(DEVICE)
x      = torch.randn(2, 200, 16, 16).to(DEVICE)
labels = torch.randint(0, 17, (2, 16, 16)).to(DEVICE)

torch.cuda.reset_peak_memory_stats()
out  = model(x)
loss = torch.nn.CrossEntropyLoss(ignore_index=0)(out, labels)
loss.backward()
print(f"Peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

Peak VRAM: 2.23 GB


In [ ]:
import torch
from models.hyperspectral_net import HyperspectralNet

DEVICE = torch.device('cuda')
model = HyperspectralNet(num_bands=200, num_classes=17, base_filters=32).to(DEVICE)
x      = torch.randn(4, 200, 16, 16).to(DEVICE)
labels = torch.randint(0, 17, (4, 16, 16)).to(DEVICE)

torch.cuda.reset_peak_memory_stats()
out  = model(x)
loss = torch.nn.CrossEntropyLoss(ignore_index=0)(out, labels)
loss.backward()
print(f"Peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

Peak VRAM: 4.39 GB


: 